In [1]:
import pandas as pd
import numpy as np
import gamspy as gs

# 1. LOAD AND PREPROCESS DATA
df = pd.read_excel("dataset_output.xlsx")
df = df.dropna(subset=['id']).copy()

ind_labels = [f"ind_{idx}" for idx in df.index]
transit_labels = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L"] 
states = ["0", "1"]  # 0: Closed, 1: Open

beta = {
    'ASC_OPT_OUT': -0.014063, 
    'ASC_A': 0.0,            
    'ASC_B': 0.035173, 
    'ASC_C': -0.049251, 
    'ASC_D': -0.045553, 
    'ASC_E': 0.233853, 
    'ASC_F': -0.305134, 
    'ASC_G': 0.269048,
    'ASC_H': 0.026862,
    'ASC_I': -0.216403,
    'ASC_J': 0.329965,
    'ASC_K': -0.105195,
    'ASC_L': -0.087918,
    'B_DIST': -0.050087,  
    'B_SHELTER': 0.639126, 
    'B_RAINY': -0.379045,      
    'B_DIST_RAINY': -0.017229
}

# 2. PRE-COMPUTE CONSTANT RATIOS WITH ESTIMATED PARAMETERS
v_optout = beta['ASC_OPT_OUT']
kappa_matrix = np.zeros((len(df), len(transit_labels), 2))
omega_matrix = np.zeros((len(df), len(transit_labels), 2))

for j_idx, letter in enumerate(transit_labels):
    dist = df[f"distance_{letter}"].values
    rainy = df["rainy"].values
    has_shelter_attr = df[f"shelter_{letter}"].values 
    asc = beta[f"ASC_{letter}"]
    
    v_base = (
        asc 
        + beta['B_DIST'] * dist 
        + beta['B_SHELTER'] * has_shelter_attr 
        + beta['B_RAINY'] * rainy 
        + beta['B_DIST_RAINY'] * dist * rainy
    )
    
    # State 0 (Closed)
    kappa_matrix[:, j_idx, 0] = 0.0001 # Small epsilon placeholder to prevent initial matrix registration errors
    omega_matrix[:, j_idx, 0] = 0.0
    
    # State 1 (Open)
    v = v_base
    kappa_matrix[:, j_idx, 1] = np.exp(v) / np.exp(v_optout)
    omega_matrix[:, j_idx, 1] = np.exp(v) / (np.exp(v_optout) + np.exp(v))

# 3. INITIALIZE GAMSPY CONTAINER & UNIVERSAL SETS
m = gs.Container()

i = gs.Set(m, name="i", records=ind_labels)
j = gs.Set(m, name="j", records=transit_labels)
s = gs.Set(m, name="s", records=states)

# Create a specific subset containing only the OPEN state to screen out division by zero
s_open = gs.Set(m, name="s_open", domain=[s], records=["1"])
kappa = gs.Parameter(m, name="kappa", domain=[i, j, s], records=kappa_matrix)
omega = gs.Parameter(m, name="omega", domain=[i, j, s], records=omega_matrix)

# 4. VARIABLES & MILP LINEAR FORMULATION
X = gs.Variable(m, name="X", domain=[i, j, s], type="Positive")      
X_tilde = gs.Variable(m, name="X_tilde", domain=[i], type="Positive") 
Y = gs.Variable(m, name="Y", domain=[j, s], type="Binary")         
open_station = gs.Variable(m, name="open_station", domain=[j], type="Binary")   
total_market_share = gs.Variable(m, name="total_market_share", type="Free")

eq_y_state0 = gs.Equation(m, name="eq_y_state0", domain=[j])
eq_y_state1 = gs.Equation(m, name="eq_y_state1", domain=[j])
eq_prob_conservation = gs.Equation(m, name="eq_prob_conservation", domain=[i])
eq_bound_a4 = gs.Equation(m, name="eq_bound_a4", domain=[i, j, s])
eq_bound_a5 = gs.Equation(m, name="eq_bound_a5", domain=[i, j, s])
eq_bound_a6 = gs.Equation(m, name="eq_bound_a6", domain=[i, j, s_open])

eq_budget = gs.Equation(m, name="eq_budget")
eq_objective = gs.Equation(m, name="eq_objective")

eq_y_state1[j] = Y[j, "1"] == open_station[j]
eq_y_state0[j] = Y[j, "0"] == 1 - open_station[j]

eq_prob_conservation[i] = X_tilde[i] + gs.Sum((j, s), X[i, j, s]) == 1

eq_bound_a4[i, j, s] = X[i, j, s] <= omega[i, j, s] * Y[j, s]
eq_bound_a5[i, j, s] = X[i, j, s] <= kappa[i, j, s] * X_tilde[i]

eq_bound_a6[i, j, s_open] = X_tilde[i] - (1 / kappa[i, j, s_open]) * X[i, j, s_open] + Y[j, s_open] <= 1

eq_budget[...] = gs.Sum(j, open_station[j]) <= 8
eq_objective[...] = total_market_share == gs.Sum((i, j, s), X[i, j, s])

# 5. EXECUTE MIP OPTIMIZATION SOLVER
station_paper_mip = gs.Model(
    m,
    name="station_paper_mip",
    equations=[
        eq_y_state0, eq_y_state1, eq_prob_conservation, 
        eq_bound_a4, eq_bound_a5, eq_bound_a6, eq_budget, eq_objective
    ],
    problem="MIP",
    sense="MAX",
    objective=total_market_share
)

station_paper_mip.solve(solver="CPLEX")
print(f"Maximized Total Transit Share: {(station_paper_mip.objective_value / 1000 * 100):.2f}%")
print("\nOptimal Station Opening Matrix:")
print(open_station.records.set_index("j")["level"])


Maximized Total Transit Share: 74.34%

Optimal Station Opening Matrix:
j
A    1.0
B    1.0
C    1.0
D    1.0
E    1.0
F    0.0
G    1.0
H    1.0
I    0.0
J    1.0
K    0.0
L    0.0
Name: level, dtype: float64
